In [14]:
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads
from shapely.ops import unary_union

# Load the CSV file
#df = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/hotels_traveltime.csv")
df = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/cores_traveltime.csv")



In [12]:
df

,Unnamed: 0,place,geometry,longitude,latitude,walking_5min,walking_15min,walking_25min,cycling_5min,cycling_15min,...,public_transport_15min_12-25,public_transport_15min_12-30,public_transport_30min_12-00,public_transport_30min_12-05,public_transport_30min_12-10,public_transport_30min_12-15,public_transport_30min_12-20,public_transport_30min_12-25,public_transport_30min_12-30,merged_isochrone
0,0,Bijlmer Arena,POINT (4.945563151367942 52.31278966366592),4.945563,52.312790,MULTIPOLYGON (((4.941169754660223 52.314014278...,MULTIPOLYGON (((4.932436295086518 52.317213505...,MULTIPOLYGON (((4.926609335234389 52.320412750...,MULTIPOLYGON (((4.93437889078632 52.3158836319...,MULTIPOLYGON (((4.902532602660358 52.326872815...,...,MULTIPOLYGON (((4.890686284634285 52.333953026...,MULTIPOLYGON (((4.917673986637965 52.348010305...,MULTIPOLYGON (((4.832937508821487 52.342608333...,MULTIPOLYGON (((4.89996122661978 52.2930638492...,MULTIPOLYGON (((4.765919626690447 52.307551964...,MULTIPOLYGON (((4.839232138299849 52.345237343...,MULTIPOLYGON (((4.7613013667287305 52.31284185...,MULTIPOLYGON (((4.757800501014572 52.303513821...,MULTIPOLYGON (((4.832937508821487 52.342608333...,MULTIPOLYGON (((4.9114014326478355 52.29888813...
1,1,Zuidas,POINT (4.873155691753168 52.339676417731134),4.873156,52.339676,MULTIPOLYGON (((4.872967077419162 52.336694314...,MULTIPOLYGON (((4.8605324265081435 52.33944415...,MULTIPOLYGON (((4.848778450919781 52.342653218...,MULTIPOLYGON (((4.856686754268594 52.340808094...,MULTIPOLYGON (((4.8238934895489365 52.33429084...,...,MULTIPOLYGON (((4.760723788058385 52.307340024...,MULTIPOLYGON (((4.756336910300888 52.307853061...,MULTIPOLYGON (((4.66420346184168 52.3042233558...,MULTIPOLYGON (((4.676008884096518 52.297108100...,MULTIPOLYGON (((4.682235082727857 52.262194111...,MULTIPOLYGON (((4.775293242651969 52.326463659...,MULTIPOLYGON (((4.703004744078498 52.286698744...,MULTIPOLYGON (((4.645212841394823 52.252857294...,MULTIPOLYGON (((4.66127473465167 52.3042233558...,MULTIPOLYGON (((4.765864240936935 52.310099789...
2,2,Buikslotermeerplein,POINT (4.938824441842034 52.39845831627722),4.938824,52.398458,MULTIPOLYGON (((4.93606971000554 52.3995204944...,MULTIPOLYGON (((4.927367716096342 52.402717358...,MULTIPOLYGON (((4.916428005089983 52.403665954...,MULTIPOLYGON (((4.925712735042907 52.402729439...,MULTIPOLYGON (((4.897467417060398 52.392878571...,...,MULTIPOLYGON (((4.9123048493638635 52.40016665...,MULTIPOLYGON (((4.922539860010147 52.404546508...,MULTIPOLYGON (((4.886765349772759 52.348820508...,MULTIPOLYGON (((4.892158851260319 52.405880918...,MULTIPOLYGON (((4.888574657787103 52.381163276...,MULTIPOLYGON (((4.886894342605956 52.408257345...,MULTIPOLYGON (((4.87730050238315 52.3375968151...,MULTIPOLYGON (((4.884058136492968 52.348428774...,MULTIPOLYGON (((4.9030869654379785 52.40453894...,MULTIPOLYGON (((4.899493811564753 52.377292008...
3,3,Sloterdijk Centrum,POINT (4.839443737103754 52.38852656323145),4.839444,52.388527,MULTIPOLYGON (((4.8372598066926 52.38657900894...,MULTIPOLYGON (((4.8313559289090335 52.38709193...,MULTIPOLYGON (((4.821151417447254 52.380405583...,MULTIPOLYGON (((4.824000182270538 52.388889289...,MULTIPOLYGON (((4.789241153688636 52.388193570...,...,MULTIPOLYGON (((4.824782863492146 52.380590416...,MULTIPOLYGON (((4.837257550680079 52.359471385...,MULTIPOLYGON (((4.636030685156584 52.381166232...,MULTIPOLYGON (((4.641940171364695 52.387454831...,MULTIPOLYGON (((4.757151232508477 52.302512510...,MULTIPOLYGON (((4.744616392767057 52.407094591...,MULTIPOLYGON (((4.603370808530599 52.355525448...,MULTIPOLYGON (((4.633525897981599 52.384726853...,MULTIPOLYGON (((4.698313380591571 52.289948988...,MULTIPOLYGON (((4.83357433276251 52.3545286344...
4,4,Osdorpplein,POINT (4.805761979295833 52.35880838651294),4.805762,52.358808,MULTIPOLYGON (((4.804711635923013 52.356288145...,MULTIPOLYGON (((4.794494330009911 52.358588610...,MULTIPOLYGON (((4.782032691698987 52.353256331...,MULTIPOLYGON (((4.789979801164

In [2]:
#hotels_tt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/hotels_test.csv")

### Create 30 min isochrones for public transport

In [24]:
# List of columns that contain isochrone geometries (update these column names!)
isochrone_columns = ["public_transport_30min_12-00", "public_transport_30min_12-05", "public_transport_30min_12-10",
                    "public_transport_30min_12-15", "public_transport_30min_12-20", "public_transport_30min_12-25", "public_transport_30min_12-30"]
# Modify as needed

# Convert all isochrone columns from WKT to Shapely geometries
for col in isochrone_columns:
    df[col] = df[col].apply(lambda x: loads(x) if pd.notna(x) else None)




In [ ]:
# Create a new column that merges all isochrones per row
df["merged_isochrone"] = df.apply(lambda row: unary_union([row[col] for col in isochrone_columns if row[col] is not None]), axis=1)

In [ ]:
# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry="merged_isochrone", crs="EPSG:4326")  # Adjust CRS if needed

# Select only necessary columns
gdf = gdf[["place_id", "name", "merged_isochrone"]]

gdf.to_file("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/merged_isochrones_pt30.geojson", driver="GeoJSON")  # Save as GeoJSON


### Create 15 min isochrones for public transport

In [15]:
# List of columns that contain isochrone geometries (update these column names!)
isochrone_columns = ["public_transport_15min_12-00", "public_transport_15min_12-05", "public_transport_15min_12-10",
                    "public_transport_15min_12-15", "public_transport_15min_12-20", "public_transport_15min_12-25", 
                     "public_transport_15min_12-30"]
# Modify as needed

# Convert all isochrone columns from WKT to Shapely geometries
for col in isochrone_columns:
    df[col] = df[col].apply(lambda x: loads(x) if pd.notna(x) else None)




In [16]:
# Create a new column that merges all isochrones per row
df["merged_isochrone"] = df.apply(lambda row: unary_union([row[col] for col in isochrone_columns if row[col] is not None]), axis=1)

In [17]:
# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry="merged_isochrone", crs="EPSG:4326")  # Adjust CRS if needed

# Select only necessary columns
#gdf = gdf[["place" "merged_isochrone"]]
#gdf = gdf[["place_id", "name", "merged_isochrone"]]

#gdf.to_file("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/cores_merged_isochrones_pt15.geojson", driver="GeoJSON")  # Save as GeoJSON


In [6]:
df

,Unnamed: 0,place,geometry,longitude,latitude,walking_5min,walking_15min,walking_25min,cycling_5min,cycling_15min,...,public_transport_15min_12-25,public_transport_15min_12-30,public_transport_30min_12-00,public_transport_30min_12-05,public_transport_30min_12-10,public_transport_30min_12-15,public_transport_30min_12-20,public_transport_30min_12-25,public_transport_30min_12-30,merged_isochrone
0,0,Bijlmer Arena,POINT (4.945563151367942 52.31278966366592),4.945563,52.312790,MULTIPOLYGON (((4.941169754660223 52.314014278...,MULTIPOLYGON (((4.932436295086518 52.317213505...,MULTIPOLYGON (((4.926609335234389 52.320412750...,MULTIPOLYGON (((4.93437889078632 52.3158836319...,MULTIPOLYGON (((4.902532602660358 52.326872815...,...,MULTIPOLYGON (((4.890686284634285 52.333953026...,MULTIPOLYGON (((4.917673986637965 52.348010305...,MULTIPOLYGON (((4.832937508821487 52.342608333...,MULTIPOLYGON (((4.89996122661978 52.2930638492...,MULTIPOLYGON (((4.765919626690447 52.307551964...,MULTIPOLYGON (((4.839232138299849 52.345237343...,MULTIPOLYGON (((4.7613013667287305 52.31284185...,MULTIPOLYGON (((4.757800501014572 52.303513821...,MULTIPOLYGON (((4.832937508821487 52.342608333...,MULTIPOLYGON (((4.9114014326478355 52.29888813...
1,1,Zuidas,POINT (4.873155691753168 52.339676417731134),4.873156,52.339676,MULTIPOLYGON (((4.872967077419162 52.336694314...,MULTIPOLYGON (((4.8605324265081435 52.33944415...,MULTIPOLYGON (((4.848778450919781 52.342653218...,MULTIPOLYGON (((4.856686754268594 52.340808094...,MULTIPOLYGON (((4.8238934895489365 52.33429084...,...,MULTIPOLYGON (((4.760723788058385 52.307340024...,MULTIPOLYGON (((4.756336910300888 52.307853061...,MULTIPOLYGON (((4.66420346184168 52.3042233558...,MULTIPOLYGON (((4.676008884096518 52.297108100...,MULTIPOLYGON (((4.682235082727857 52.262194111...,MULTIPOLYGON (((4.775293242651969 52.326463659...,MULTIPOLYGON (((4.703004744078498 52.286698744...,MULTIPOLYGON (((4.645212841394823 52.252857294...,MULTIPOLYGON (((4.66127473465167 52.3042233558...,MULTIPOLYGON (((4.765864240936935 52.310099789...
2,2,Buikslotermeerplein,POINT (4.938824441842034 52.39845831627722),4.938824,52.398458,MULTIPOLYGON (((4.93606971000554 52.3995204944...,MULTIPOLYGON (((4.927367716096342 52.402717358...,MULTIPOLYGON (((4.916428005089983 52.403665954...,MULTIPOLYGON (((4.925712735042907 52.402729439...,MULTIPOLYGON (((4.897467417060398 52.392878571...,...,MULTIPOLYGON (((4.9123048493638635 52.40016665...,MULTIPOLYGON (((4.922539860010147 52.404546508...,MULTIPOLYGON (((4.886765349772759 52.348820508...,MULTIPOLYGON (((4.892158851260319 52.405880918...,MULTIPOLYGON (((4.888574657787103 52.381163276...,MULTIPOLYGON (((4.886894342605956 52.408257345...,MULTIPOLYGON (((4.87730050238315 52.3375968151...,MULTIPOLYGON (((4.884058136492968 52.348428774...,MULTIPOLYGON (((4.9030869654379785 52.40453894...,MULTIPOLYGON (((4.899493811564753 52.377292008...
3,3,Sloterdijk Centrum,POINT (4.839443737103754 52.38852656323145),4.839444,52.388527,MULTIPOLYGON (((4.8372598066926 52.38657900894...,MULTIPOLYGON (((4.8313559289090335 52.38709193...,MULTIPOLYGON (((4.821151417447254 52.380405583...,MULTIPOLYGON (((4.824000182270538 52.388889289...,MULTIPOLYGON (((4.789241153688636 52.388193570...,...,MULTIPOLYGON (((4.824782863492146 52.380590416...,MULTIPOLYGON (((4.837257550680079 52.359471385...,MULTIPOLYGON (((4.636030685156584 52.381166232...,MULTIPOLYGON (((4.641940171364695 52.387454831...,MULTIPOLYGON (((4.757151232508477 52.302512510...,MULTIPOLYGON (((4.744616392767057 52.407094591...,MULTIPOLYGON (((4.603370808530599 52.355525448...,MULTIPOLYGON (((4.633525897981599 52.384726853...,MULTIPOLYGON (((4.698313380591571 52.289948988...,MULTIPOLYGON (((4.83357433276251 52.3545286344...
4,4,Osdorpplein,POINT (4.805761979295833 52.35880838651294),4.805762,52.358808,MULTIPOLYGON (((4.804711635923013 52.356288145...,MULTIPOLYGON (((4.794494330009911 52.358588610...,MULTIPOLYGON (((4.782032691698987 52.353256331...,MULTIPOLYGON (((4.789979801164